# 📚 Sistem Rekomendasi Buku — Hybrid (Collaborative Filtering + Content-Based)
Dataset: **Book-Crossing** (`BX-Users`, `BX-Books`, `BX-Book-Ratings`)

**Alur notebook**
1. Konfigurasi path & load data
2. Preprocessing (Users, Books, Ratings) + ringkasan sebelum/sesudah
3. EDA singkat
4. Membangun matriks interaksi
5. Model: CF (PureSVD) + Content-Based (TF-IDF) → Hybrid
6. Evaluasi leave-one-out & pemilihan bobot hybrid
7. Demo rekomendasi (user lama, user baru, buku serupa)
8. Simpan artefak untuk dashboard Streamlit

In [ ]:
import os, json, time, re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

pd.set_option("display.max_colwidth", 70)
plt.rcParams["figure.figsize"] = (8, 4)

# ===================== PATH DATA (Kaggle) =====================
KAGGLE_DIR = Path("/kaggle/input/datasets/lailadyah/book-adl")
RATINGS_CSV = KAGGLE_DIR / "BX-Book-Ratings.csv"
BOOKS_CSV   = KAGGLE_DIR / "BX-Books.csv"
USERS_CSV   = KAGGLE_DIR / "BX-Users.csv"

# fallback bila dijalankan di komputer lokal (folder ./data)
if not KAGGLE_DIR.exists():
    KAGGLE_DIR = Path("data")
    RATINGS_CSV, BOOKS_CSV, USERS_CSV = (KAGGLE_DIR / n for n in
        ["BX-Book-Ratings.csv", "BX-Books.csv", "BX-Users.csv"])

OUT = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("artifacts")
OUT.mkdir(parents=True, exist_ok=True)

# ===================== HYPER-PARAMETER =====================
MIN_USER_INTERACTIONS = 5     # user minimal 5 interaksi positif
MIN_ITEM_INTERACTIONS = 5     # buku minimal 5 interaksi
POSITIVE_MIN_RATING   = 5     # rating eksplisit 1-4 = negatif -> dibuang
IMPLICIT_WEIGHT       = 1.0   # bobot rating 0 (interaksi tanpa nilai)
N_FACTORS   = 64              # dimensi laten SVD
N_EVAL_USERS = 3000
TOP_K = 10
SEED  = 42
rng = np.random.default_rng(SEED)

for p in (RATINGS_CSV, BOOKS_CSV, USERS_CSV):
    print(("OK      " if p.exists() else "TIDAK ADA"), p)

## 1. Load Data Mentah

In [ ]:
def read_bx(path):
    return pd.read_csv(path, sep=";", encoding="latin-1", quotechar='"',
                       escapechar="\\", on_bad_lines="skip", low_memory=False)

users_raw   = read_bx(USERS_CSV)
books_raw   = read_bx(BOOKS_CSV)
ratings_raw = read_bx(RATINGS_CSV)

users_raw.columns   = ["user_id", "location", "age"]
books_raw.columns   = ["isbn", "title", "author", "year", "publisher", "img_s", "img_m", "img_l"]
ratings_raw.columns = ["user_id", "isbn", "rating"]

print("Users  :", users_raw.shape)
print("Books  :", books_raw.shape)
print("Ratings:", ratings_raw.shape)
display(users_raw.head(3)); display(books_raw.head(3)); display(ratings_raw.head(3))

## 2. Preprocessing
**Users** — usia tidak valid (<5 atau >100) → NaN; negara/kota dari kolom `location`.
**Books** — tahun di luar 1900–2026 → NaN; perbaiki karakter rusak (`cÃ¡mara` → `cámara`); buang ISBN duplikat & judul kosong.
**Ratings** — buang duplikat; hanya buku yang punya metadata; rating 0 = implisit, 1–4 = negatif (dibuang), 5–10 = positif.

In [ ]:
def fix_mojibake(x):
    try:
        return x.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return x

summary = {}   # untuk tabel sebelum/sesudah

# ---------- USERS ----------
users = users_raw.copy()
users["age"] = pd.to_numeric(users["age"], errors="coerce")
summary["Usia tidak valid -> NaN"] = int(((users["age"] < 5) | (users["age"] > 100)).sum())
users.loc[(users["age"] < 5) | (users["age"] > 100), "age"] = np.nan
loc = users["location"].fillna("").str.split(",")
users["country"] = loc.str[-1].str.strip().replace("", np.nan).fillna("unknown")
users["city"] = loc.str[0].str.strip()

# ---------- BOOKS ----------
books = books_raw.copy()
books["year"] = pd.to_numeric(books["year"], errors="coerce")
summary["Tahun terbit tidak valid -> NaN"] = int(((books["year"] < 1900) | (books["year"] > 2026)).sum())
books.loc[(books["year"] < 1900) | (books["year"] > 2026), "year"] = np.nan
for c in ["title", "author", "publisher"]:
    books[c] = (books[c].fillna("").astype(str).map(fix_mojibake)
                .str.replace("&amp;", "&", regex=False).str.strip())
n0 = len(books)
books = books.drop_duplicates("isbn")
books = books[books["title"] != ""]
summary["Buku dibuang (duplikat/judul kosong)"] = n0 - len(books)

# ---------- RATINGS ----------
ratings = ratings_raw.copy()
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["rating", "user_id"])
ratings["user_id"] = ratings["user_id"].astype(int)
ratings["isbn"] = ratings["isbn"].astype(str).str.strip()
n1 = len(ratings)
ratings = ratings.drop_duplicates(["user_id", "isbn"])
summary["Rating duplikat dibuang"] = n1 - len(ratings)
n2 = len(ratings)
ratings = ratings[ratings["isbn"].isin(books["isbn"])]
summary["Rating tanpa metadata buku dibuang"] = n2 - len(ratings)

print(pd.Series(summary).to_frame("jumlah").to_string())
print(f"\nSetelah cleaning -> users={len(users):,} | books={len(books):,} | ratings={len(ratings):,}")

## 3. EDA Singkat

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ratings["rating"].astype(int).value_counts().sort_index().plot.bar(ax=ax[0], color="#4C78A8")
ax[0].set_title("Distribusi rating (0 = implisit)")
users["age"].dropna().plot.hist(bins=40, ax=ax[1], color="#F58518")
ax[1].set_title("Distribusi usia user")
users["country"].value_counts().head(10).sort_values().plot.barh(ax=ax[2], color="#54A24B")
ax[2].set_title("10 negara dengan user terbanyak")
plt.tight_layout(); plt.show()

n_u, n_i = ratings.user_id.nunique(), ratings.isbn.nunique()
print(f"Rating implisit (0): {(ratings.rating==0).mean():.1%}")
print(f"Rating eksplisit   : {(ratings.rating>0).sum():,}")
print(f"Sparsity           : {1 - len(ratings)/(n_u*n_i):.5%}")

## 4. Matriks Interaksi
- rating **0** → bobot 1 (implisit)  
- rating **5–10** → bobot `1 + (r-4)/3` (1,33 … 3,0)  
- rating **1–4** → dibuang  
- filter iteratif: user ≥ 5 dan buku ≥ 5 interaksi

In [ ]:
def build_interactions(r):
    r = r[(r["rating"] == 0) | (r["rating"] >= POSITIVE_MIN_RATING)].copy()
    r["w"] = np.where(r["rating"] == 0, IMPLICIT_WEIGHT, 1.0 + (r["rating"] - 4) / 3.0)
    while True:
        n0 = len(r)
        uc = r["user_id"].value_counts()
        r = r[r["user_id"].isin(uc[uc >= MIN_USER_INTERACTIONS].index)]
        ic = r["isbn"].value_counts()
        r = r[r["isbn"].isin(ic[ic >= MIN_ITEM_INTERACTIONS].index)]
        if len(r) == n0:
            break
    return r.reset_index(drop=True)

inter = build_interactions(ratings)
u_ids = np.sort(inter["user_id"].unique())
i_ids = np.sort(inter["isbn"].unique())
u2i = {u: k for k, u in enumerate(u_ids)}
i2i = {i: k for k, i in enumerate(i_ids)}

X = sparse.csr_matrix((inter["w"].values.astype(np.float32),
                       (inter["user_id"].map(u2i).values, inter["isbn"].map(i2i).values)),
                      shape=(len(u_ids), len(i_ids)))
print(f"Interaksi positif: {len(inter):,} | user: {len(u_ids):,} | buku: {len(i_ids):,}")
print(f"Ukuran matriks   : {X.shape} | density: {X.nnz / (X.shape[0]*X.shape[1]):.4%}")

# tabel item (metadata + statistik)
explicit = ratings[ratings["rating"] > 0]
items = books.set_index("isbn").loc[i_ids].reset_index()
ex = explicit.groupby("isbn")["rating"].agg(["mean", "count"])
items = items.merge(ex, left_on="isbn", right_index=True, how="left")
items["n_inter"] = np.asarray((X > 0).sum(0)).ravel()
items["n_explicit"] = items["count"].fillna(0)
items["avg_rating"] = items["mean"]
gm = explicit["rating"].mean()
items["bayes_rating"] = (items["n_explicit"] * items["mean"].fillna(gm) + 10 * gm) / (items["n_explicit"] + 10)
items = items[["isbn", "title", "author", "year", "publisher", "img_m",
               "n_inter", "n_explicit", "avg_rating", "bayes_rating"]]
items.head()

## 5. Model Hybrid
**(a) CF — PureSVD**: matriks user×buku dinormalisasi per user (÷√jumlah interaksi) lalu `TruncatedSVD`. Profil user dihitung dengan *fold-in* `(x·V)·Vᵀ`, sehingga berlaku juga untuk user baru.  
**(b) Content-Based — TF-IDF** atas *judul + penulis (×2) + penerbit*.  
**(c) Hybrid** = `w_cf · minmax(CF) + w_content · minmax(Content)`.

In [ ]:
def normalize_users(X):
    deg = np.asarray((X > 0).sum(1)).ravel().astype(np.float32)
    return sparse.diags(1.0 / np.sqrt(np.maximum(deg, 1))) @ X

def fit_svd(X):
    svd = TruncatedSVD(n_components=N_FACTORS, n_iter=7, random_state=SEED)
    svd.fit(normalize_users(X))
    return svd.components_.T.astype(np.float32)          # (n_items, k)

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def cf_scores(x, V):
    xn = x / np.sqrt(max(int((x > 0).sum()), 1))
    return np.asarray((xn @ V) @ V.T).ravel()

def content_scores(x, T):
    return np.asarray(((x @ T) @ T.T).todense()).ravel()

def hybrid(cf, ct, w):
    return w[0] * minmax(cf) + w[1] * minmax(ct)

# ---- TF-IDF konten
text = (items["title"] + " " + items["author"] + " " + items["author"] + " " + items["publisher"]).str.lower()
tfidf = TfidfVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2),
                        max_features=60000, sublinear_tf=True)
T = normalize(tfidf.fit_transform(text)).astype(np.float32).tocsr()
print("TF-IDF matrix:", T.shape)

## 6. Evaluasi (Leave-One-Out)
Satu buku positif tiap user disembunyikan, model dilatih tanpa buku itu, lalu dilihat peringkat buku tersembunyi di antara semua buku yang belum dibaca. Metrik: **HitRate@10, NDCG@10, MRR**.

In [ ]:
def evaluate(X, T, w_grid):
    eligible = np.where(np.diff(X.indptr) >= 6)[0]
    eval_users = rng.choice(eligible, size=min(N_EVAL_USERS, len(eligible)), replace=False)

    Xtr = X.tolil(copy=True); held = {}
    for u in eval_users:
        its = X.indices[X.indptr[u]:X.indptr[u + 1]]
        h = rng.choice(its); held[u] = h; Xtr[u, h] = 0
    Xtr = Xtr.tocsr(); Xtr.eliminate_zeros()
    V = fit_svd(Xtr)                                      # latih TANPA item tersembunyi

    models = {"CF (SVD)": (1.0, 0.0), "Content-Based": (0.0, 1.0)}
    for w in w_grid:
        models[f"Hybrid {w}"] = w
    res = {m: dict(hit=0, ndcg=0, mrr=0) for m in models}

    for u in eval_users:
        x = Xtr[u]; seen = x.indices
        cf, ct = cf_scores(x, V), content_scores(x, T)
        for name, w in models.items():
            s = hybrid(cf, ct, w); s[seen] = -np.inf
            rank = int((s > s[held[u]]).sum())
            if rank < TOP_K:
                res[name]["hit"] += 1; res[name]["ndcg"] += 1 / np.log2(rank + 2)
            res[name]["mrr"] += 1 / (rank + 1)
    n = len(eval_users)
    df = pd.DataFrame([{"model": m, f"HitRate@{TOP_K}": v["hit"]/n,
                        f"NDCG@{TOP_K}": v["ndcg"]/n, "MRR": v["mrr"]/n}
                       for m, v in res.items()])
    return df, models

w_grid = [(round(float(a), 1), round(float(1 - a), 1)) for a in np.arange(0.1, 1.0, 0.1)]
metrics, models = evaluate(X, T, w_grid)
display(metrics.round(4))

hy = metrics[metrics.model.str.startswith("Hybrid")]
best = hy.sort_values(f"NDCG@{TOP_K}", ascending=False).iloc[0]["model"]
best_w = models[best]
print("Bobot hybrid terbaik (CF, Content):", best_w)

metrics.set_index("model")[f"HitRate@{TOP_K}"].plot.barh(color="#4C78A8")
plt.title(f"HitRate@{TOP_K} per model"); plt.tight_layout(); plt.show()

## 7. Model Final & Fungsi Rekomendasi

In [ ]:
V = fit_svd(X)                      # SVD final pada seluruh data
Vn = normalize(V)

def norm_title(t):
    t = re.sub(r"\(.*?\)|\[.*?\]", " ", str(t).lower())
    return re.sub(r"[^a-z0-9]+", " ", t).strip()

def user_vector(user_id):
    u = u2i.get(int(user_id))
    return None if u is None else X[u]

def vector_from_isbns(picks):
    idx, val = [], []
    for isbn, r in picks.items():
        j = i2i.get(isbn)
        if j is None: continue
        idx.append(j); val.append(1.0 if r == 0 else 1.0 + (max(r, 5) - 4) / 3.0)
    return sparse.csr_matrix((val, ([0]*len(idx), idx)), shape=(1, len(i2i)), dtype=np.float32)

def recommend(x, k=10, weights=None, max_per_author=2):
    w = weights or best_w
    cf, ct = cf_scores(x, V), content_scores(x, T)
    score = hybrid(cf, ct, w)
    score[x.indices] = -np.inf                                   # sudah dibaca
    seen_titles = {norm_title(t) for t in items["title"].values[x.indices]}
    picked, per_author = [], {}
    for j in np.argsort(-score)[: k * 15]:
        a, nt = items.at[j, "author"], norm_title(items.at[j, "title"])
        if per_author.get(a, 0) >= max_per_author or nt in seen_titles: continue
        per_author[a] = per_author.get(a, 0) + 1; seen_titles.add(nt); picked.append(j)
        if len(picked) == k: break
    res = items.iloc[picked][["isbn", "title", "author", "year", "publisher"]].copy()
    res["score"] = score[picked]
    res["s_cf"], res["s_content"] = minmax(cf)[picked], minmax(ct)[picked]
    return res.reset_index(drop=True)

def similar_books(isbn, k=10, alpha=0.3):
    j = i2i[isbn]
    s = alpha * minmax(Vn @ Vn[j]) + (1 - alpha) * minmax(np.asarray((T @ T[j].T).todense()).ravel())
    s[j] = -np.inf
    seen, out = {norm_title(items.at[j, "title"])}, []
    for c in np.argsort(-s)[: k * 15]:
        nt = norm_title(items.at[c, "title"])
        if nt in seen: continue
        seen.add(nt); out.append(c)
        if len(out) == k: break
    res = items.iloc[out][["isbn", "title", "author", "year"]].copy()
    res["similarity"] = s[out]
    return res.reset_index(drop=True)

def search_books(q, limit=10):
    q = q.lower()
    m = items["title"].str.lower().str.contains(q, regex=False) | items["author"].str.lower().str.contains(q, regex=False)
    return items[m].sort_values("n_inter", ascending=False).head(limit)[["isbn", "title", "author", "n_inter"]]

### Demo 1 — Rekomendasi untuk user yang sudah ada

In [ ]:
uid = int(inter.groupby("user_id").size().sort_values(ascending=False).index[50])
x = user_vector(uid)
print(f"User {uid} | jumlah interaksi: {x.nnz}")
print("\nContoh riwayat:"); display(items.iloc[x.indices][["title", "author"]].head(5))
print("Rekomendasi:"); display(recommend(x, k=10).round(3))

### Demo 2 — User baru (pilih buku sendiri)

In [ ]:
display(search_books("grisham", 5))
isbn_pilih = search_books("the firm", 1).iloc[0].isbn
picks = {isbn_pilih: 9}                           # {ISBN: rating 0-10}
display(recommend(vector_from_isbns(picks), k=10).round(3))

### Demo 3 — Buku serupa

In [ ]:
display(similar_books(isbn_pilih, k=8).round(3))

## 8. Simpan Artefak untuk Dashboard
File `model.joblib` ini dipakai oleh `app.py` (Streamlit). Unduh dari panel **Output** Kaggle (`/kaggle/working/artifacts/`).

In [ ]:
ut = users.set_index("user_id").reindex(u_ids)
ut["n_inter"] = np.diff(X.indptr)
ut = ut.reset_index()[["user_id", "country", "city", "age", "n_inter"]]

inter_c = inter.merge(users[["user_id", "country"]], on="user_id")
country_pop = inter_c.groupby(["country", "isbn"])["w"].sum().reset_index()
country_pop = country_pop[country_pop.country.isin(inter_c["country"].value_counts().head(25).index)]

eda = {
    "n_users": int(len(users)), "n_books": int(len(books)), "n_ratings": int(len(ratings)),
    "pct_implicit": float((ratings["rating"] == 0).mean()),
    "rating_dist": ratings["rating"].astype(int).value_counts().sort_index().to_dict(),
    "sparsity": float(1 - len(ratings) / (n_u * n_i)),
    "n_explicit": int(len(explicit)),
    "model_users": len(u_ids), "model_items": len(i_ids), "model_inter": int(X.nnz),
    "age_hist": users["age"].dropna().astype(int).value_counts().sort_index().to_dict(),
    "top_countries": users["country"].value_counts().head(15).to_dict(),
    "year_hist": books["year"].dropna().astype(int).value_counts().sort_index().to_dict(),
    "top_authors": inter.merge(items[["isbn", "author"]], on="isbn")["author"].value_counts().head(15).to_dict(),
}

joblib.dump({"X": X, "V": V, "T": T, "items": items, "users": ut,
             "u2i": u2i, "i2i": i2i, "u_ids": u_ids, "i_ids": i_ids,
             "best_w": best_w, "country_pop": country_pop, "eda": eda,
             "metrics": metrics, "top_k": TOP_K}, OUT / "model.joblib", compress=3)
print("Tersimpan:", OUT / "model.joblib", f"({(OUT / 'model.joblib').stat().st_size/1e6:.1f} MB)")